<a href="https://colab.research.google.com/github/Sathishssds73/rag_chatbot/blob/main/Rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-text-splitters
!pip install sentence-transformers faiss-cpu chromadb
!pip install groq pypdf pymupdf

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Kiruthiga Resume (2).pdf")
documents = loader.load()

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=20
)

char_docs = char_splitter.split_documents(documents)

# PRINT OUTPUT
for i, chunk in enumerate(char_docs[:10]):  # show first 10
    print(f"\n🔹 Chunk {i+1}:")
    print(chunk.page_content)
    print("-" * 50)

print("Total chunks:", len(char_docs))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

recursive_docs = recursive_splitter.split_documents(documents)

# PRINT OUTPUT
for i, chunk in enumerate(recursive_docs[:10]):
    print(f"\n🔹 Recursive Chunk {i+1}:")
    print(chunk.page_content)
    print("-" * 50)

print("Total chunks:", len(recursive_docs))

In [ ]:
!pip install faiss-cpu sentence-transformers

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

In [ ]:
vectorstore = FAISS.from_documents(
    char_docs,      # 👈 your chunks
    embedding
)

In [ ]:
# For recursive
vectorstore = FAISS.from_documents(recursive_docs, embedding)


In [ ]:
vectorstore.save_local("resume_vector_db")

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
query = "What are the skills?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n🔹 Result {i+1}:")
    print(doc.page_content)
    print("-" * 50)

In [ ]:
print("Total vectors stored:", len(vectorstore.index_to_docstore_id))

In [ ]:
pip install streamlit

In [ ]:
!pip install langchain-groq

In [ ]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant"   # ✅ FIXED MODEL
)

def chat_with_resume(query):
    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a resume assistant.
Answer ONLY using the context below.
If not found, say "Java,python,sql".

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
while True:
    user_input = input("\n🧑 Ask something about the resume: ")

    if user_input.lower() in ["exit", "quit"]:
        print("👋 Exiting chatbot...")
        break

    answer = chat_with_resume(user_input)

    print("\n🤖 Answer:")
    print(answer)

In [ ]:
import os
print(os.getcwd())           # shows current directory
print(os.listdir('.'))       # lists all files in that directory


In [ ]:
import json

# Replace with whatever name showed up in os.listdir()
notebook_filename = "your_actual_filename.ipynb"

with open(notebook_filename, "r") as f:
    nb = json.load(f)

nb["metadata"].pop("widgets", None)

with open(notebook_filename, "w") as f:
    json.dump(nb, f, indent=1)

print("Done! Widget metadata removed.")